In [ ]:
'''
Step 1  Top-30 SHAP
Step 2  Missingness
Step 3  Correlation
Step 4  Redundancy

Step 5  Train/test + imputation
Step 6  Reduced XGBoost training
Step 7  Threshold review
Step 8 Full vs reduced comparison
Step 9 Save deployment model 

# Step 10 verification
#step 11 Prediction on new data

'''

'\nStep 1  Top-30 SHAP\nStep 2  Missingness\nStep 3  Correlation\nStep 4  Redundancy\n\nStep 5  Train/test + imputation\nStep 6  Reduced XGBoost training\nStep 7  Threshold review\nStep 8 Full vs reduced comparison\nStep 9 Save deployment model \n\n'

In [95]:
import pandas as pd

top30_shap = pd.read_csv(
    "../model_dir/artifacts_dir/top30_shap_features.csv"
)

print(top30_shap)

                      Feature  Mean_ABS_SHAP
0                    num__age       0.362456
1                num__BUN_max       0.249174
2               num__BUN_mean       0.208940
3               num__sao2_std       0.144781
4         num__heartrate_mean       0.110318
5            num__lactate_min       0.107137
6               num__sao2_min       0.104624
7        num__respiration_std       0.078170
8       num__respiration_mean       0.075553
9                 num__pH_min       0.064077
10               num__RDW_min       0.053832
11              num__FiO2_min       0.053729
12         num__heartrate_min       0.053476
13       num__AST (SGOT)_mean       0.046135
14   num__hospitaladmitoffset       0.046036
15         num__heartrate_max       0.045531
16         num__heartrate_std       0.039430
17          num__calcium_mean       0.038953
18               num__RDW_max       0.033982
19          num__lactate_mean       0.031233
20       num__admissionweight       0.028556
21        

In [96]:
top30_shap["Raw_Feature"] = top30_shap["Feature"].str.replace(
    "num__",
    "",
    regex=False
)

print(top30_shap)

                      Feature  Mean_ABS_SHAP           Raw_Feature
0                    num__age       0.362456                   age
1                num__BUN_max       0.249174               BUN_max
2               num__BUN_mean       0.208940              BUN_mean
3               num__sao2_std       0.144781              sao2_std
4         num__heartrate_mean       0.110318        heartrate_mean
5            num__lactate_min       0.107137           lactate_min
6               num__sao2_min       0.104624              sao2_min
7        num__respiration_std       0.078170       respiration_std
8       num__respiration_mean       0.075553      respiration_mean
9                 num__pH_min       0.064077                pH_min
10               num__RDW_min       0.053832               RDW_min
11              num__FiO2_min       0.053729              FiO2_min
12         num__heartrate_min       0.053476         heartrate_min
13       num__AST (SGOT)_mean       0.046135       AST (SGOT)_

steps:

In [97]:
# step1: Step 1 — Load raw merged dataset
df = pd.read_csv(
    "../final_preprocessed_raw.csv"
)

print("Dataset shape:", df.shape)

print("-"*100)

top30_features = top30_shap["Raw_Feature"].tolist()

for feature in top30_features:
    print(feature, "->", feature in df.columns)

Dataset shape: (2426, 611)
----------------------------------------------------------------------------------------------------
age -> True
BUN_max -> True
BUN_mean -> True
sao2_std -> True
heartrate_mean -> True
lactate_min -> True
sao2_min -> True
respiration_std -> True
respiration_mean -> True
pH_min -> True
RDW_min -> True
FiO2_min -> True
heartrate_min -> True
AST (SGOT)_mean -> True
hospitaladmitoffset -> True
heartrate_max -> True
heartrate_std -> True
calcium_mean -> True
RDW_max -> True
lactate_mean -> True
admissionweight -> True
albumin_mean -> True
albumin_min -> True
st1_std -> True
cvp_mean -> True
lactate_max -> True
FiO2_mean -> True
total bilirubin_mean -> True
total protein_min -> True
RBC_std -> True


In [98]:
# Step2: Missingness check 
missing_results = []
missingness_selected = []

for feature in top30_features:

    missing_percentage = df[feature].isna().mean() * 100

    # Store missing percentage of every feature
    missing_results.append({
        "Feature": feature,
        "Missing_Percentage": round(missing_percentage, 2)
    })

    # Select features with missing percentage <= 40%
    if missing_percentage <= 40:
        missingness_selected.append(feature)


# Display missing percentage of all Top 30 features
missing_df = pd.DataFrame(missing_results)

missing_df = missing_df.sort_values(
    by="Missing_Percentage",
    ascending=False
)

print("Missing Percentage of Top 30 Features")
print(missing_df)

print("_" * 100)

# Display selected features
print("Selected Features (Missing Percentage <= 40%)")
print("Features remaining:", len(missingness_selected))

print("\nSelected Feature Names:")
print(missingness_selected)

Missing Percentage of Top 30 Features
                 Feature  Missing_Percentage
24              cvp_mean               89.82
29               RBC_std               82.69
25           lactate_max               80.83
19          lactate_mean               80.83
5            lactate_min               80.83
9                 pH_min               74.57
11              FiO2_min               69.54
26             FiO2_mean               69.54
28     total protein_min               62.00
27  total bilirubin_mean               61.83
13       AST (SGOT)_mean               61.38
21          albumin_mean               57.34
22           albumin_min               57.34
23               st1_std               56.47
18               RDW_max               27.95
10               RDW_min               27.95
17          calcium_mean               17.97
1                BUN_max               14.96
2               BUN_mean               14.96
7        respiration_std               12.70
8       respirati

In [99]:


'''
Missing <= 40%
       ↓
Keep as candidate

Missing > 40%
       ↓
Do not keep as deployment candidate

########################################

BUN_max       14.96%  → KEEP
sao2_min       5.52%  → KEEP
RDW_max       27.95%  → KEEP

albumin_mean  57.34%  → REMOVE/EXCLUDE
FiO2_mean     69.54%  → REMOVE/EXCLUDE
lactate_min   80.83%  → REMOVE/EXCLUDE
cvp_mean      89.82%  → REMOVE/EXCLUDE



########################################


**********************************************************************************************
Important interview point

40% is not a standard universal rule.


“I examined feature missingness and used 40% as a project-specific screening threshold for the 
reduced deployment model. Features with higher missingness were excluded to improve practical 
availability and reduce dependence on imputation.
**************************************************************************************************
'''


'\nMissing <= 40%\n       ↓\nKeep as candidate\n\nMissing > 40%\n       ↓\nDo not keep as deployment candidate\n\n########################################\n\nBUN_max       14.96%  → KEEP\nsao2_min       5.52%  → KEEP\nRDW_max       27.95%  → KEEP\n\nalbumin_mean  57.34%  → REMOVE/EXCLUDE\nFiO2_mean     69.54%  → REMOVE/EXCLUDE\nlactate_min   80.83%  → REMOVE/EXCLUDE\ncvp_mean      89.82%  → REMOVE/EXCLUDE\n\n\n\n########################################\n\n\n**********************************************************************************************\nImportant interview point\n\n40% is not a standard universal rule.\n\n\n“I examined feature missingness and used 40% as a project-specific screening threshold for the \nreduced deployment model. Features with higher missingness were excluded to improve practical \navailability and reduce dependence on imputation.\n**************************************************************************************************\n'

In [100]:
# Step 3: Correlation check

print("Features after missingness check:")
print(missingness_selected)
print("Number of features:", len(missingness_selected))

print("*" * 120)

# Calculate absolute correlation
corr = df[missingness_selected].corr().abs()

print("Correlation Matrix:")
print(corr)


# Identify highly correlated feature pairs
high_corr_df = corr.stack().reset_index()

high_corr_df.columns = ["Feature_1", "Feature_2", "Correlation"]

high_corr_df = high_corr_df[
    (high_corr_df["Correlation"] > 0.90) &
    (high_corr_df["Feature_1"] < high_corr_df["Feature_2"])
]

high_corr_df["Correlation"] = high_corr_df["Correlation"].round(3)

print("\nHighly Correlated Feature Pairs:")
print(high_corr_df)

Features after missingness check:
['age', 'BUN_max', 'BUN_mean', 'sao2_std', 'heartrate_mean', 'sao2_min', 'respiration_std', 'respiration_mean', 'RDW_min', 'heartrate_min', 'hospitaladmitoffset', 'heartrate_max', 'heartrate_std', 'calcium_mean', 'RDW_max', 'admissionweight']
Number of features: 16
************************************************************************************************************************
Correlation Matrix:
                          age   BUN_max  BUN_mean  sao2_std  heartrate_mean  \
age                  1.000000  0.247333  0.267586  0.066585        0.146544   
BUN_max              0.247333  1.000000  0.991236  0.065808        0.077441   
BUN_mean             0.267586  0.991236  1.000000  0.067794        0.072651   
sao2_std             0.066585  0.065808  0.067794  1.000000        0.032761   
heartrate_mean       0.146544  0.077441  0.072651  0.032761        1.000000   
sao2_min             0.103842  0.106349  0.109029  0.801758        0.083292   
respir

In [101]:
'''
NOTE

# Why correlation


Top SHAP features = 30
        ↓
Missingness check
        ↓
16 remaining

Now we ask:

Are some of these 16 features giving nearly the same information?

That's where correlation comes in.

For example:

BUN_max
BUN_mean

Suppose correlation calculation gives:

BUN_max ↔ BUN_mean = 0.95

They move very similarly.

So correlation is the measurement/check.

****************************************************************************************************************
REDUNDANCY
****************************************************************************************************************
4. Then what is redundancy?

Redundancy is the conclusion we may make from that correlation.

Think:

Correlation = tool/check
Redundancy  = problem we are trying to detect

Simple example:

Height_cm     Height_m
----------------------
170           1.70
180           1.80
190           1.90

Correlation:

Height_cm ↔ Height_m ≈ 1.0

Therefore:

These two features contain essentially
the same information
        ↓
REDUNDANT

We don't need both.


################################################### END ###############################################################

1) So don't think:

Redundancy
    ↓
Correlation
 
2)                  It's actually doing 

                   Correlation check
                        ↓
                   Find very strongly related features
                        ↓
                   Potential redundancy
                        ↓
                   Decide whether one can be removed

3) For this project:

BUN_max ───────┐
               │
               ├─ calculate correlation
               │
BUN_mean ──────┘

        ↓

Suppose correlation = 0.95

        ↓

Potentially redundant

        ↓

Check SHAP + clinical meaning

        ↓

BUN_max SHAP  = 0.249
BUN_mean SHAP = 0.209

        ↓

Potential choice: keep BUN_max


'''

"\nNOTE\n\n# Why correlation\n\n\nTop SHAP features = 30\n        ↓\nMissingness check\n        ↓\n16 remaining\n\nNow we ask:\n\nAre some of these 16 features giving nearly the same information?\n\nThat's where correlation comes in.\n\nFor example:\n\nBUN_max\nBUN_mean\n\nSuppose correlation calculation gives:\n\nBUN_max ↔ BUN_mean = 0.95\n\nThey move very similarly.\n\nSo correlation is the measurement/check.\n\n****************************************************************************************************************\nREDUNDANCY\n****************************************************************************************************************\n4. Then what is redundancy?\n\nRedundancy is the conclusion we may make from that correlation.\n\nThink:\n\nCorrelation = tool/check\nRedundancy  = problem we are trying to detect\n\nSimple example:\n\nHeight_cm     Height_m\n----------------------\n170           1.70\n180           1.80\n190           1.90\n\nCorrelation:\n\nHeight_cm ↔ He

In [102]:

'''
STEP4 Redundency  LOGIC:

Highly correlated pair
        ↓
Compare their SHAP importance
        ↓
Keep higher SHAP
Remove lower SHAP

'''

'\nSTEP4 Redundency  LOGIC:\n\nHighly correlated pair\n        ↓\nCompare their SHAP importance\n        ↓\nKeep higher SHAP\nRemove lower SHAP\n\n'

In [103]:
# Step 4-1: Create SHAP dictionary

shap_scores = {}

for index, row in top30_shap.iterrows():

    feature = row["Raw_Feature"]
    score = row["Mean_ABS_SHAP"]

    shap_scores[feature] = score


# Step 4-2: Remove redundant features using SHAP scores

redundancy_selected = missingness_selected.copy()

for index, row in high_corr_df.iterrows():

    feature_1 = row["Feature_1"]
    feature_2 = row["Feature_2"]

    shap_1 = shap_scores[feature_1]
    shap_2 = shap_scores[feature_2]

    if shap_1 >= shap_2:
        keep_feature = feature_1
        remove_feature = feature_2
    else:
        keep_feature = feature_2
        remove_feature = feature_1

    print("Keep:", keep_feature, "| Remove:", remove_feature)

    if remove_feature in redundancy_selected:
        redundancy_selected.remove(remove_feature)


# Step 4-3: Display features after redundancy removal

print("\nFeatures after redundancy review:")
print("Total features:", len(redundancy_selected))
print(redundancy_selected)


# Step 4-4: Clinical / inference availability check

final_features = redundancy_selected.copy()

print("\nFinal deployment features:", len(final_features))

for feature in final_features:
    print(feature)

Keep: BUN_max | Remove: BUN_mean
Keep: RDW_min | Remove: RDW_max

Features after redundancy review:
Total features: 14
['age', 'BUN_max', 'sao2_std', 'heartrate_mean', 'sao2_min', 'respiration_std', 'respiration_mean', 'RDW_min', 'heartrate_min', 'hospitaladmitoffset', 'heartrate_max', 'heartrate_std', 'calcium_mean', 'admissionweight']

Final deployment features: 14
age
BUN_max
sao2_std
heartrate_mean
sao2_min
respiration_std
respiration_mean
RDW_min
heartrate_min
hospitaladmitoffset
heartrate_max
heartrate_std
calcium_mean
admissionweight


In [104]:
#Step 5  Train/test + imputation


from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

# Step 7 - Prepare X and y

X = df[final_features].copy()

y = df["hospitaldischargestatus"].map({
    "Alive": 0,
    "Expired": 1
})

# remove rows where target is missing
valid_rows = y.notna()

X = X[valid_rows]
y = y[valid_rows]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (2426, 14)
y shape: (2426,)


In [105]:
from sklearn.model_selection import train_test_split

X = df[final_features]

y = df["hospitaldischargestatus"].map({
    "Alive": 0,
    "Expired": 1
})

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (1940, 14)
X_test: (486, 14)
y_train: (1940,)
y_test: (486,)


In [106]:
# preprocessing 

from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median")

X_train_processed = imputer.fit_transform(X_train)

X_test_processed = imputer.transform(X_test)

print(X_train_processed.shape)
print(X_test_processed.shape)

(1940, 14)
(486, 14)


In [107]:
# Step 6  Reduced XGBoost training

 

#Use the  previous tuned parameters from your full model:

from xgboost import XGBClassifier

reduced_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.01,
    subsample=0.9,
    colsample_bytree=1.0,
    scale_pos_weight=8,
    random_state=42,
    eval_metric="logloss"
)

reduced_model.fit(
    X_train_processed,
    y_train
)

print("Reduced model training completed")

print("*_*_"*100)

# Basic Performance check

from sklearn.metrics import accuracy_score, roc_auc_score

y_pred = reduced_model.predict(X_test_processed)

y_prob = reduced_model.predict_proba(
    X_test_processed
)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Reduced model training completed
*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_*_
Accuracy: 0.8600823045267489
ROC-AUC: 0.7932754182754184


In [108]:
# Step 7 —  Threshold review - > Check thresholds


from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

thresholds = [0.50, 0.45, 0.40, 0.35, 0.30, 0.25]

for threshold in thresholds:

    y_pred_threshold = (y_prob >= threshold).astype(int)

    print("\nThreshold:", threshold)
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred_threshold))

    print("Precision:", precision_score(y_test, y_pred_threshold))
    print("Recall:", recall_score(y_test, y_pred_threshold))
    print("F1:", f1_score(y_test, y_pred_threshold))


Threshold: 0.5
Confusion Matrix:
[[396  48]
 [ 20  22]]
Precision: 0.3142857142857143
Recall: 0.5238095238095238
F1: 0.39285714285714285

Threshold: 0.45
Confusion Matrix:
[[384  60]
 [ 20  22]]
Precision: 0.2682926829268293
Recall: 0.5238095238095238
F1: 0.3548387096774194

Threshold: 0.4
Confusion Matrix:
[[371  73]
 [ 16  26]]
Precision: 0.26262626262626265
Recall: 0.6190476190476191
F1: 0.36879432624113473

Threshold: 0.35
Confusion Matrix:
[[348  96]
 [ 12  30]]
Precision: 0.23809523809523808
Recall: 0.7142857142857143
F1: 0.35714285714285715

Threshold: 0.3
Confusion Matrix:
[[315 129]
 [ 11  31]]
Precision: 0.19375
Recall: 0.7380952380952381
F1: 0.3069306930693069

Threshold: 0.25
Confusion Matrix:
[[272 172]
 [ 11  31]]
Precision: 0.15270935960591134
Recall: 0.7380952380952381
F1: 0.2530612244897959


In [109]:
'''
This result actually looks reasonable for our mortality-screening goal.

At 0.35:

TN = 348     FP = 96
FN = 12      TP = 30

Recall    = 71.43%
Precision = 23.81%
F1        = 35.71%

This result actually looks reasonable for our mortality-screening goal.

At 0.35:

TN = 348     FP = 96
FN = 12      TP = 30

Recall    = 71.43%
Precision = 23.81%
F1        = 35.71%


                 Threshold 0.50    Threshold 0.35
Recall               52.38%            71.43%  ✅
False Negatives          20                12  ✅
True Positives            22                30  ✅
False Positives           48                96  ❌

'''

'\nThis result actually looks reasonable for our mortality-screening goal.\n\nAt 0.35:\n\nTN = 348     FP = 96\nFN = 12      TP = 30\n\nRecall    = 71.43%\nPrecision = 23.81%\nF1        = 35.71%\n\nThis result actually looks reasonable for our mortality-screening goal.\n\nAt 0.35:\n\nTN = 348     FP = 96\nFN = 12      TP = 30\n\nRecall    = 71.43%\nPrecision = 23.81%\nF1        = 35.71%\n\n\n                 Threshold 0.50    Threshold 0.35\nRecall               52.38%            71.43%  ✅\nFalse Negatives          20                12  ✅\nTrue Positives            22                30  ✅\nFalse Positives           48                96  ❌\n\n'

Step 8 — Compare Full vs Reduced Model


Your main comparison is now:

| Metric        | Full model | Reduced 14-feature model |
| ------------- | ---------: | -----------------------: |
| Features      |        572 |                   **14** |
| ROC-AUC       |     ~0.837 |                **0.793** |
| Recall @ 0.35 |     ~69.0% |                **71.4%** |
| FN @ 0.35     |         13 |                   **12** |


Note :  The key result is strong for your deployment story:

We reduced the raw input requirement from 572 features to 14, while ROC-AUC decreased by about 0.044, and the reduced model still achieved 71.4% mortality recall at the selected 0.35 threshold.

In [110]:
# Step 9 — Save the deployment package.

import joblib

deployment_package = {
    "model": reduced_model,
    "imputer": imputer,
    "features": final_features,
    "threshold": 0.35
}

joblib.dump(
    deployment_package,
    "../model_dir/deployment_icu_mortality_model.pkl"
)

print("Deployment model saved successfully")

Deployment model saved successfully


In [111]:
# Step 10 verification

loaded_package = joblib.load(
    "../model_dir/deployment_icu_mortality_model.pkl"
)

print(loaded_package.keys())
print("Number of features:", len(loaded_package["features"]))
print("Threshold:", loaded_package["threshold"])

dict_keys(['model', 'imputer', 'features', 'threshold'])
Number of features: 14
Threshold: 0.35


#step 11 Prediction on new data

In [112]:
# Step 12 - Predict on New Patient Data

import joblib
import pandas as pd


# Load deployment package
package = joblib.load(
    "../model_dir/deployment_icu_mortality_model.pkl"
)

model = package["model"]
imputer = package["imputer"]
features = package["features"]
threshold = package["threshold"]


def predict_mortality(input_data):

    # Convert input into one-row DataFrame
    if isinstance(input_data, pd.Series):
        input_df = input_data.to_frame().T
    else:
        input_df = pd.DataFrame(
            [input_data]
        )

    # Keep correct 14 features and correct order
    input_df = input_df[features]

    # Apply same imputation used during training
    input_processed = imputer.transform(
        input_df
    )

    # Mortality probability
    probability = model.predict_proba(
        input_processed
    )[:, 1][0]

    # Apply threshold
    if probability >= threshold:
        prediction = "Expired"
    else:
        prediction = "Alive"

    return prediction, probability

In [113]:
test_patient = X_test.iloc[0]
print(test_patient)

prediction, probability = predict_mortality(
    test_patient
)
print("_"*100)
print("Prediction:", prediction)

print("_"*100)
print(
    "Mortality Probability:",
    round(probability, 4)
)

print("_"*100)
print(
    "Actual:",
    "Expired"
    if y_test.iloc[0] == 1
    else "Alive"
)

print("_"*100)
print(" Both Prediction and actual same")

age                     76.000000
BUN_max                  5.000000
sao2_std                 1.920296
heartrate_mean          73.765799
sao2_min                90.000000
respiration_std          3.515125
respiration_mean        21.178439
RDW_min                 14.100000
heartrate_min           59.000000
hospitaladmitoffset      0.000000
heartrate_max           96.000000
heartrate_std            7.166528
calcium_mean             8.600000
admissionweight        100.200000
Name: 1483, dtype: float64
____________________________________________________________________________________________________
Prediction: Alive
____________________________________________________________________________________________________
Mortality Probability: 0.1126
____________________________________________________________________________________________________
Actual: Alive
____________________________________________________________________________________________________
 Both Prediction and actual sam

In [114]:
test_input = test_patient.to_dict()

print(test_input)

{'age': 76.0, 'BUN_max': 5.0, 'sao2_std': 1.920296369236452, 'heartrate_mean': 73.76579925650557, 'sao2_min': 90.0, 'respiration_std': 3.515124540923292, 'respiration_mean': 21.17843866171004, 'RDW_min': 14.1, 'heartrate_min': 59.0, 'hospitaladmitoffset': 0.0, 'heartrate_max': 96.0, 'heartrate_std': 7.166528061208512, 'calcium_mean': 8.6, 'admissionweight': 100.2}
